# Notebook 5: Interpretation, Recommendation Engine & Ethical Analysis
## CS336 - Artificial Intelligence and Machine Learning (AIML)
### Assignment: Smart Energy Consumption Advisory Agent

**Purpose:** This final notebook:
1. **Interprets** cluster and anomaly results in human-readable language.
2...

In [ ]:
# ─── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
print('Imports successful.')

## 1. Load Final Dataset

In [ ]:
# ─── Load the fully modelled dataset from Notebook 3 ─────────────────────────
df = pd.read_csv('data_modelled.csv', parse_dates=['timestamp'], index_col='timestamp')
daily = pd.read_csv('data_daily.csv', parse_dates=['timestamp'], index_col='timestamp')

print(f'Dataset: {len(df):,} rows | Anomalies: {df["is_anomaly"].sum():,}')

## 2. Cluster Interpretation

Based on the centroid analysis in Notebook 3, we label each cluster with a  
human-readable behavioural description and assign a recommendation category.

In [ ]:
# ─── Cluster label mapping ───────────────────────────────────────────────────
# Each label is derived from the centroid's average power and time-of-day distribution
CLUSTER_LABELS = {
    0: {
        'name': 'Low Off-Peak Usage',
        'description': 'Very low consumption, primarily late night/early morning.',
        'risk': 'low',
        'advice_key': 'maintain'
    },
    1: {
        'name': 'Morning Ramp-Up',
        'description': 'Consumption rises sharply from 06:00–09:00 (kitchen + laundry).',
        'risk': 'medium',
        'advice_key': 'shift_morning'
    },
    2: {
        'name': 'Steady Daytime',
        'description': 'Moderate, relatively flat usage throughout working hours.',
        'risk': 'low',
        'advice_key': 'automate'
    },
    3: {
        'name': 'High Evening Peak',
        'description': 'Highest usage 17:00–22:00 — HVAC, cooking, entertainment.',
        'risk': 'high',
        'advice_key': 'reduce_peak'
    }
}

# Map labels onto the dataframe for readability
df['cluster_name'] = df['cluster'].map(lambda c: CLUSTER_LABELS[c]['name'])

print('Cluster label distribution:')
print(df['cluster_name'].value_counts())

## 3. Recommendation Engine

The engine accepts a **current usage snapshot** (from a live or recent reading) and returns  
prioritised, actionable advice based on the cluster, anomaly status, and time-of-use period.

In [ ]:
# ─── Rule-based recommendation engine ────────────────────────────────────────

# Advice library: keyed by advice_key from CLUSTER_LABELS
ADVICE_LIBRARY = {
    'maintain': [
        '✅ Your energy usage is low and efficient — keep it up!',
        '💡 Consider scheduling EV charging now (low-tariff window).',
        '🌙 Good time to run dishwashers or washing machines.'
    ],
    'shift_morning': [
        '⏰ Morning demand spike detected. Delay washing machine/dishwasher to after 10:00.',
        '🍳 Use the microwave instead of the oven for breakfast cooking.',
        '☀️ Consider pre-heating your home with overnight off-peak energy.'
    ],
    'automate': [
        '🤖 Automate lighting using smart plugs — unnecessary lights waste ~15% of daytime power.',
        '🌡️ Set HVAC to eco-mode during working hours if the home is empty.',
        '📊 Your daytime usage is moderate — no immediate action required.'
    ],
    'reduce_peak': [
        '🚨 High peak-hour consumption detected (17:00–22:00).',
        '🍽️ Shift oven usage to before 17:00 or after 21:00 to avoid peak tariffs.',
        '❄️ Pre-cool/pre-heat your home before 17:00 and coast through peak hours.',
        '📺 Reduce standby power: unplug entertainment systems when not in use.'
    ],
    'anomaly': [
        '⚠️ ANOMALY DETECTED: Unusually high (or low) consumption recorded.',
        '🔍 Check for appliances left running unintentionally (oven, HVAC, pool pump).',
        '🛠️ Persistent anomalies may indicate a faulty appliance — consider inspection.'
    ]
}


def generate_recommendations(cluster_id: int, is_anomaly: bool,
                              hour: int, power_kw: float) -> dict:
    """
    Generate prioritised energy-saving recommendations for a household.

    Parameters
    ----------
    cluster_id : int   — K-Means cluster assignment for the current interval
    is_anomaly : bool  — True if Isolation Forest flagged this reading
    hour       : int   — Hour of day (0-23)
    power_kw   : float — Current global active power in kW

    Returns
    -------
    dict with keys: cluster_name, risk_level, tips, anomaly_warning
    """
    info = CLUSTER_LABELS[cluster_id]

    # Retrieve base advice for this cluster
    tips = list(ADVICE_LIBRARY[info['advice_key']])

    # Append anomaly-specific advice if flagged
    anomaly_warning = None
    if is_anomaly:
        anomaly_warning = ADVICE_LIBRARY['anomaly']
        tips = anomaly_warning + tips  # Anomaly advice takes priority

    return {
        'cluster_name':    info['name'],
        'risk_level':      info['risk'],
        'current_power':   f'{power_kw:.2f} kW',
        'time_of_day':     f'{hour:02d}:00',
        'tips':            tips,
        'anomaly_warning': anomaly_warning is not None
    }


# ─── Demo: Generate advice for a high-peak evening reading ───────────────────
demo_row = df[df['cluster'] == 3].iloc[10]  # Pick a high-peak cluster row

recs = generate_recommendations(
    cluster_id=int(demo_row['cluster']),
    is_anomaly=bool(demo_row['is_anomaly']),
    hour=demo_row.name.hour,
    power_kw=demo_row['global_active_power']
)

print('=== Smart Energy Advisory Report ===')
print(f"Cluster      : {recs['cluster_name']}")
print(f"Risk Level   : {recs['risk_level'].upper()}")
print(f"Current Power: {recs['current_power']}")
print(f"Time         : {recs['time_of_day']}")
print(f"Anomaly Alert: {recs['anomaly_warning']}")
print('\nRecommendations:')
for tip in recs['tips']:
    print(f'  {tip}')

## 4. Batch Advisory Report

Apply the engine to all readings and compute aggregate statistics.

In [ ]:
# ─── Compute risk-level frequency across the full year ───────────────────────
risk_map = {0: 'low', 1: 'medium', 2: 'low', 3: 'high'}  # From CLUSTER_LABELS
df['risk_level'] = df['cluster'].map(risk_map)

risk_counts = df['risk_level'].value_counts()

# ─── Bar chart of risk-level distribution ────────────────────────────────────
colours = {'low': '#4CAF50', 'medium': '#FF9800', 'high': '#F44336'}

plt.figure(figsize=(8, 5))
bars = plt.bar(risk_counts.index,
               risk_counts.values,
               color=[colours[r] for r in risk_counts.index],
               edgecolor='white')

for bar, val in zip(bars, risk_counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 200,
             f'{val:,}', ha='center', va='bottom', fontsize=11)

plt.title('Annual Reading Count by Risk Level', fontsize=13)
plt.xlabel('Risk Level')
plt.ylabel('Number of 15-min Intervals')
plt.tight_layout()
plt.show()

# Compute estimated wasted energy in high-risk peak slots
high_risk_power = df[df['risk_level'] == 'high']['global_active_power'].mean()
low_risk_power  = df[df['risk_level'] == 'low']['global_active_power'].mean()
high_risk_count = risk_counts.get('high', 0)

potential_saving_kwh = (high_risk_power - low_risk_power) * high_risk_count * (15/60)
print(f'\nEstimated peak-period excess energy (vs. low-risk baseline): {potential_saving_kwh:.0f} kWh/year')

## 5. Ethical and Societal Analysis

Deploying AI-powered energy advisory agents raises important ethical questions.

### 5.1 Privacy and Surveillance

Smart meter data collected at 15-minute intervals can reveal **intimate behavioural patterns**:  
when residents wake up, sleep, cook, and leave home. This constitutes personal data under  
GDPR and similar legislation.

**Mitigations:**
- Data should remain **on-de...

In [ ]:
# ─── Summary: Key statistics across all notebooks ────────────────────────────
print('===== SMART ENERGY ADVISORY AGENT — FINAL SUMMARY =====')
print(f'Total records analysed   : {len(df):,}')
print(f'Anomalies detected       : {df["is_anomaly"].sum():,} ({df["is_anomaly"].mean()*100:.1f}%)')
print(f'Clusters formed          : {df["cluster"].nunique()}')
print(f'High-risk interval count : {(df["risk_level"]=="high").sum():,}')
print(f'Potential savings        : ~{potential_saving_kwh:.0f} kWh/year')
print()
print('Ethical areas addressed  : Privacy, Fairness, Transparency, Societal Benefit')
print('Notebooks completed      : 5 of 5')